# AAG-FP metadata analysis

This notebook profiles the AAG-FP metadata before downloading images. It treats `Category` as the dataset label, and keeps strict category membership separate from keyword matches in `folder_ID`.

The analysis is descriptive only: it does not claim the scraped floorplans, inferred graphs, or any image subset are human-verified ground truth.

In [ ]:
from collections import Counter
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

# Set AAG_FP_METADATA if the CSV is elsewhere, e.g. on SOC.
metadata_path = Path(os.environ.get(
    'AAG_FP_METADATA',
    Path.home() / 'Downloads' / 'AAG-FP_metadata.csv',
))

if not metadata_path.exists():
    raise FileNotFoundError(
        f'Metadata CSV not found: {metadata_path}. Set AAG_FP_METADATA to its path.'
    )

df = pd.read_csv(metadata_path)
required_columns = {'folder_ID', 'img_ID', 'img_url', 'project_url', 'Category'}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f'Missing expected columns: {sorted(missing)}')

df['Category'] = df['Category'].fillna('').str.strip()
df.head()

In [ ]:
summary = pd.Series({
    'metadata rows': len(df),
    'unique image URLs': df['img_url'].nunique(),
    'duplicate image-URL rows': int(df['img_url'].duplicated().sum()),
    'unique projects': df['project_url'].nunique(),
    'unique folders': df['folder_ID'].nunique(),
    'distinct Category labels': df['Category'].nunique(),
    'blank Category labels': int((df['Category'] == '').sum()),
}, name='value')
summary.to_frame()

## Category value counts

`Category` is the label column. The full table below is useful for selecting an initial residential candidate pool.

In [ ]:
category_counts = (
    df['Category']
    .value_counts(dropna=False)
    .rename_axis('Category')
    .reset_index(name='image_count')
)
category_counts['share_of_metadata'] = category_counts['image_count'] / len(df)
display(category_counts.style.format({'share_of_metadata': '{:.1%}'}))

print(f"Strict Apartments label: {(df['Category'] == 'Apartments').sum():,} images")

In [ ]:
top_n = 25
plot_data = category_counts.head(top_n).sort_values('image_count')
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(plot_data['Category'], plot_data['image_count'], color='#2563eb')
ax.set_title(f'Top {top_n} AAG-FP category labels')
ax.set_xlabel('Metadata rows / image records')
ax.set_ylabel('Category')
plt.tight_layout()
plt.show()

## Residential candidate definitions

Use the strict label for a clean apartment-only analysis. The keyword-based set is a *retrieval candidate pool*, not a clean class label: it can contain mixed-use projects and should be reviewed before use.

In [ ]:
strict_apartments = df[df['Category'].eq('Apartments')].copy()
folder_mentions_apartment = df[
    df['folder_ID'].fillna('').str.contains('apartment', case=False, regex=False)
].copy()

residential_categories = {
    'Apartments', 'Houses', 'Housing', 'Residential', 'Social Housing',
    'Loft', 'Penthouse', 'Dorms',
}
residential_candidates = df[df['Category'].isin(residential_categories)].copy()

candidate_summary = pd.DataFrame([
    {'subset': 'Strict Category = Apartments', 'image_records': len(strict_apartments), 'unique_projects': strict_apartments['project_url'].nunique()},
    {'subset': 'folder_ID contains apartment', 'image_records': len(folder_mentions_apartment), 'unique_projects': folder_mentions_apartment['project_url'].nunique()},
    {'subset': 'Broad residential candidate categories', 'image_records': len(residential_candidates), 'unique_projects': residential_candidates['project_url'].nunique()},
])
candidate_summary

In [ ]:
# Inspect which labels enter the broad candidate pool.
(residential_candidates['Category']
 .value_counts()
 .rename_axis('Category')
 .reset_index(name='image_count'))

## Optional exports

These CSVs contain metadata and URLs only. Do not treat them as downloaded images or graph labels.

In [ ]:
output_dir = Path('../analysis_outputs')
output_dir.mkdir(parents=True, exist_ok=True)

category_counts.to_csv(output_dir / 'aag_fp_category_counts.csv', index=False)
strict_apartments.to_csv(output_dir / 'aag_fp_strict_apartments.csv', index=False)
residential_candidates.to_csv(output_dir / 'aag_fp_residential_candidates.csv', index=False)

print(f'Wrote analysis CSVs to: {output_dir.resolve()}')